# Demo: Fusion_PSSM1110 测试集混淆矩阵

针对 **最佳方案（1110 维 PSSM 融合 BERT，Fusion_PSSM1110）**，在 **测试集** 上输出混淆矩阵与 AUC/ACC/AUPRC。

- **数据与训练逻辑与 `anticrispr_demo.ipynb` 一致**：同一数据加载（`load_anticrispr_with_ids` + 1110 维 PSSM 缓存 + `attach_pssm_features`）、同一 `FusionTrainConfig` 显式参数、划分与训练流程与 `run_finetune_with_pssm` 相同（frozen → unfrozen 两阶段、EarlyStopping、验证集最优阈值）。
- **每次运行均从头训练** Fusion_PSSM1110，不加载已有权重；训练后在验证集上选最优阈值，在测试集上输出混淆矩阵与 AUC/ACC/AUPRC。

In [1]:
# 与 anticrispr_demo.ipynb 一致：依赖、路径、数据加载（1110 维 PSSM）、预训练模型
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, average_precision_score
from tensorflow import keras

from proteinbert import (
    load_anticrispr_with_ids,
    load_pretrained_model,
    FusionTrainConfig,
    load_feature_cache,
    attach_pssm_features,
)
from proteinbert.pssm_fusion import _build_late_fusion_model, _encode_x, find_best_threshold

PROJECT_ROOT = '/home/nemophila/projects/protein_bert'
BENCHMARKS_DIR = f'{PROJECT_ROOT}/anticrispr_benchmarks'
WORK_ROOT = f'{PROJECT_ROOT}/pssm_work'
FEAT_DIR = f'{WORK_ROOT}/features'
SEED = 22

train_base_df, test_base_df = load_anticrispr_with_ids(BENCHMARKS_DIR, benchmark_name='anticrispr_binary')
variant = '1110'
parquet_path = f'{WORK_ROOT}/features/pssm_features_{variant}.parquet'
csv_path = f'{WORK_ROOT}/features/pssm_features_{variant}.csv'
cache_path = parquet_path if os.path.exists(parquet_path) else csv_path
if not os.path.exists(cache_path):
    raise FileNotFoundError(f'PSSM cache not found: {cache_path}')
feature_df, feature_cols = load_feature_cache(cache_path)
train_df = attach_pssm_features(train_base_df, feature_df, feature_cols)
test_df = attach_pssm_features(test_base_df, feature_df, feature_cols)
y_test = test_df['label'].astype(int).to_numpy()

pmg, enc = load_pretrained_model(
    local_model_dump_dir=f'{PROJECT_ROOT}/proteinbert_models',
    download_model_dump_if_not_exists=True,
    validate_downloading=False,
)
print('train_df:', train_df.shape, 'test_df:', test_df.shape, 'feature_cols:', len(feature_cols))

2026-02-22 16:41:17.771764: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


train_df: (1107, 1113) test_df: (286, 1113) feature_cols: 1110


## 训练 Fusion_PSSM1110，得到测试集预测概率与最优阈值

In [2]:
# 与 anticrispr_demo 一致：FusionTrainConfig 显式参数，划分与训练流程同 run_finetune_with_pssm
cfg = FusionTrainConfig(
    seq_len=512,
    batch_size=8,
    frozen_epochs=6,
    unfrozen_epochs=12,
    frozen_lr=1e-4,
    unfrozen_lr=2e-5,
    pssm_dropout=0.3,
    global_dropout=0.3,
    pssm_hidden_dim=128,
    global_hidden_dim=128,
    global_bottleneck_dim=64,
    fusion_hidden_dim=128,
    use_hidden_global_concat=True,
)

rng_train, rng_valid = train_test_split(
    train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED
)
x_train = rng_train[feature_cols].to_numpy(dtype=np.float32)
x_valid = rng_valid[feature_cols].to_numpy(dtype=np.float32)
x_test = test_df[feature_cols].to_numpy(dtype=np.float32)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)
x_test = scaler.transform(x_test)

y_train = rng_train['label'].astype(int).to_numpy()
y_valid = rng_valid['label'].astype(int).to_numpy()

X_train = _encode_x(enc, rng_train['seq'].tolist(), cfg.seq_len, x_train)
X_valid = _encode_x(enc, rng_valid['seq'].tolist(), cfg.seq_len, x_valid)
X_test = _encode_x(enc, test_df['seq'].tolist(), cfg.seq_len, x_test)

print('Training Fusion_PSSM1110 (seed=22)...')
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=cfg.patience, restore_best_weights=True
    )
]
model = _build_late_fusion_model(
    pmg, seq_len=cfg.seq_len, pssm_dim=len(feature_cols),
    freeze_pretrained_layers=True, cfg=cfg,
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.frozen_lr),
    loss='binary_crossentropy',
)
model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=cfg.frozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)
for layer in model.layers:
    layer.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.unfrozen_lr),
    loss='binary_crossentropy',
)
model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=cfg.unfrozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)

valid_prob = model.predict(X_valid, batch_size=cfg.batch_size, verbose=0).reshape(-1)
best_thr = find_best_threshold(y_valid, valid_prob)
y_prob_test = model.predict(X_test, batch_size=cfg.batch_size, verbose=0).reshape(-1)
y_pred_test = (y_prob_test >= best_thr).astype(int)
print(f'Best threshold (valid): {best_thr:.3f}')

Training Fusion_PSSM1110 (seed=22)...


2026-02-22 16:41:19.364948: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-02-22 16:41:19.365953: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-02-22 16:41:19.386019: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:2a:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-02-22 16:41:19.386049: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-02-22 16:41:19.388021: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-02-22 16:41:19.388130: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-02-2

Best threshold (valid): 0.250


## 测试集混淆矩阵

In [3]:
auc = roc_auc_score(y_test, y_prob_test)
acc = accuracy_score(y_test, y_pred_test)
auprc = average_precision_score(y_test, y_prob_test)
print('Fusion_PSSM1110 (seed=22) — Test set metrics')
print(f'  AUC:   {auc:.4f}')
print(f'  ACC:   {acc:.4f}')
print(f'  AUPRC: {auprc:.4f}')

cm = confusion_matrix(y_test, y_pred_test)
labels = ['Non-Acr', 'Acr']
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = 'True'
cm_df.columns.name = 'Predicted'
print('\nConfusion matrix')
display(cm_df)

Fusion_PSSM1110 (seed=22) — Test set metrics
  AUC:   0.9436
  ACC:   0.8986
  AUPRC: 0.7307

Confusion matrix


Predicted,Non-Acr,Acr
True,,
Non-Acr,235,25
Acr,4,22
